# Notebook 04 — Đánh giá caption

Đọc `dataset.json` (notebook 02) và các file caption `captions_*.json` (notebook 03), rồi tính:

| Chỉ số | Áp dụng cho | Cách tính |
|---|---|---|
| **CIDEr** [19] | COCO, SciCap+ | Thư viện `pycocoevalcap`, so với câu tham chiếu |
| **CLIPScore** [21] | Mọi ảnh | Tự viết theo bài gốc: 2,5 × max(cos(câu, ảnh), 0), CLIP ViT-B/32, tiền tố "A photo depicts" |
| **RefCLIPScore** [21] | COCO, SciCap+ | Trung bình điều hòa giữa CLIPScore và độ giống nhất với câu tham chiếu |
| **Lỗi sai dữ kiện** | Biểu đồ tự vẽ | Tự viết: so số liệu, xu hướng, cột cao/thấp nhất với đáp án |
| Độ dài, thời gian | Mọi ảnh | Thống kê |

GPU giúp tính CLIPScore nhanh hơn nhưng không bắt buộc.

## 1. Cấu hình và thư viện

In [ ]:
import os, json, glob
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ImageCaptioning"
else:
    BASE = "/content"
DATA_DIR = f"{BASE}/data"
PRED_DIR = f"{BASE}/ketqua"
os.makedirs(PRED_DIR, exist_ok=True)

!pip install -q pycocoevalcap
# CIDEr dùng bộ tách từ PTB chạy bằng Java
!java -version 2>&1 | head -1 || (apt-get -qq update && apt-get -qq install -y default-jre-headless)

In [ ]:
import re
with open(f"{DATA_DIR}/dataset.json", encoding="utf8") as f:
    dataset = json.load(f)
items = {item["id"]: item for item in dataset["items"]}

# Bỏ tiền tố "Figure 3:" / "Fig. 6" còn sót trong caption tham chiếu của SciCap+
FIG_PREFIX = re.compile(r"^\s*(?:Figure|Fig\.?)\s*\d+[a-zA-Z]?\s*[:.]?\s+", re.IGNORECASE)
for item in items.values():
    if item["source"] == "scicap_plus_test":
        item["references"] = [FIG_PREFIX.sub("", r).strip() for r in item["references"]]
print(dataset["meta"]["counts"])

predictions = {}
for path in sorted(glob.glob(f"{PRED_DIR}/captions_*.json")):
    with open(path, encoding="utf8") as f:
        run = json.load(f)
    predictions[f"{run['model']} | {run['config']}"] = run
print("Các lần chạy:", list(predictions))

## 2. CIDEr

In [ ]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider

tokenizer = PTBTokenizer()


def cider_scores(run, source):
    ids = [i for i in run["captions"] if items.get(i, {}).get("source") == source and items[i].get("references")]
    if not ids:
        return None, {}
    gts = tokenizer.tokenize({i: [{"caption": c} for c in items[i]["references"]] for i in ids})
    res = tokenizer.tokenize({i: [{"caption": run["captions"][i]["text"]}] for i in ids})
    # Cider tính trọng số IDF từ chính tập tham chiếu đang đánh giá
    corpus, per_image = Cider().compute_score(gts, res)
    return corpus, dict(zip(gts.keys(), per_image))


cider = {}
for name, run in predictions.items():
    for source in ("coco_val2017", "scicap_plus_test"):
        corpus, per_image = cider_scores(run, source)
        if corpus is not None:
            cider[(name, source)] = (corpus, per_image)
            print(f"{name:45s} {source:18s} CIDEr = {corpus:.3f} ({len(per_image)} ảnh)")

## 3. CLIPScore và RefCLIPScore

In [ ]:
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
W = 2.5  # hệ số theo bài [21]


@torch.no_grad()
def embed_texts(texts):
    inputs = clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
    return torch.nn.functional.normalize(clip.get_text_features(**inputs), dim=-1)


@torch.no_grad()
def embed_image(path):
    inputs = clip_proc(images=Image.open(path).convert("RGB"), return_tensors="pt").to(device)
    return torch.nn.functional.normalize(clip.get_image_features(**inputs), dim=-1)


image_cache = {}
clip_rows = {}
for name, run in predictions.items():
    for item_id, pred in run["captions"].items():
        item = items[item_id]
        if item_id not in image_cache:
            image_cache[item_id] = embed_image(os.path.join(DATA_DIR, item["file"]))
        cand = embed_texts(["A photo depicts " + pred["text"]])
        clip_s = W * max((cand @ image_cache[item_id].T).item(), 0.0)
        ref_clip_s = None
        if item.get("references"):
            # Theo code gốc của [21], câu tham chiếu cũng được thêm tiền tố
            refs = embed_texts(["A photo depicts " + r for r in item["references"]])
            ref_sim = max((cand @ refs.T).max().item(), 0.0)
            ref_clip_s = 2 * clip_s * ref_sim / (clip_s + ref_sim) if clip_s + ref_sim > 0 else 0.0
        clip_rows[(name, item_id)] = (clip_s, ref_clip_s)
print(len(clip_rows), "cặp (lần chạy, ảnh) đã chấm")

## 4. Lỗi sai dữ kiện trên biểu đồ tự vẽ

In [ ]:
import re

UP = ("increas", "rise", "rising", "rose", "grow", "upward", "climb", "higher each")
DOWN = ("decreas", "declin", "drop", "fall", "fell", "downward", "lower each", "reduc")
MIXED = ("fluctuat", "varies", "varied", "up and down", "rises and falls", "then drops", "then decreas", "then increas")
NUMBER = re.compile(r"(?<![\w.])(\d+(?:[.,]\d+)?)\s*(%|percent)?")


def numbers_in(text):
    """Các số xuất hiện trong caption, bỏ qua số nằm trong tên riêng như "BLIP-2"."""
    found = []
    for match in NUMBER.finditer(text):
        start = match.start()
        if start > 0 and text[start - 1] == "-":
            continue
        found.append(float(match.group(1).replace(",", ".")))
    return found


def check_chart(caption, truth, tolerance=0.5):
    """So caption với đáp án của biểu đồ tự vẽ; trả về các ý đúng, sai và bị bỏ qua."""
    text = caption.lower()
    values = [float(v) for v in truth["values"]]
    axis_numbers = {float(x) for x in truth["categories"] if x.isdigit()}
    result = {"numbers_correct": [], "numbers_wrong": [], "trend": None, "extreme": None}

    for number in numbers_in(caption):
        if number in axis_numbers:
            continue
        if any(abs(number - v) <= tolerance for v in values):
            result["numbers_correct"].append(number)
        else:
            result["numbers_wrong"].append(number)

    if truth["chart_type"] == "line":
        said_up = any(w in text for w in UP)
        said_down = any(w in text for w in DOWN)
        said_mixed = any(w in text for w in MIXED) or (said_up and said_down)
        if said_up or said_down or said_mixed:
            said = "mixed" if said_mixed else "increasing" if said_up else "decreasing"
            result["trend"] = "correct" if said == truth["trend"] else "wrong"

    for word, key in (("highest", "max"), ("largest", "max"), ("peak", "max"), ("lowest", "min"), ("smallest", "min")):
        if word in text:
            idx = text.index(word)
            window = text[max(0, idx - 60): idx + 60]
            named = [c for c in truth["categories"] if c.lower() in window]
            if named:
                result["extreme"] = "correct" if truth[key] in named else "wrong"
            break

    result["errors"] = len(result["numbers_wrong"]) + (result["trend"] == "wrong") + (result["extreme"] == "wrong")
    result["facts_stated"] = (len(result["numbers_correct"]) + len(result["numbers_wrong"])
                              + (result["trend"] is not None) + (result["extreme"] is not None))
    return result


fact_rows = {}
for name, run in predictions.items():
    for item_id, pred in run["captions"].items():
        if items[item_id]["source"] == "synthetic_chart":
            fact_rows[(name, item_id)] = check_chart(pred["text"], items[item_id])
print(len(fact_rows), "caption biểu đồ đã chấm")

## 5. Bảng tổng hợp

In [ ]:
import pandas as pd

rows = []
for name, run in predictions.items():
    for item_id, pred in run["captions"].items():
        item = items[item_id]
        clip_s, ref_clip_s = clip_rows.get((name, item_id), (None, None))
        cider_value = cider.get((name, item["source"]), (None, {}))[1].get(item_id)
        facts = fact_rows.get((name, item_id), {})
        rows.append({
            "run": name, "model": run["model"], "config": run["config"], "id": item_id, "source": item["source"],
            "values_labelled": item.get("values_labelled"), "caption": pred["text"],
            "words": len(pred["text"].split()), "seconds": pred.get("seconds"),
            "cider": cider_value, "clip_s": clip_s, "ref_clip_s": ref_clip_s,
            "facts_stated": facts.get("facts_stated"), "fact_errors": facts.get("errors"),
            "numbers_wrong": facts.get("numbers_wrong"),
        })
df = pd.DataFrame(rows)
df.to_csv(f"{PRED_DIR}/scores_per_item.csv", index=False)

summary = df.groupby(["run", "source"]).agg(
    images=("id", "count"), words=("words", "mean"), seconds=("seconds", "mean"),
    clip_s=("clip_s", "mean"), ref_clip_s=("ref_clip_s", "mean"),
    facts_stated=("facts_stated", "sum"), fact_errors=("fact_errors", "sum")).reset_index()
summary["cider"] = [cider.get((r.run, r.source), (None,))[0] for r in summary.itertuples()]
summary.to_csv(f"{PRED_DIR}/scores_summary.csv", index=False)
pd.set_option("display.width", 200)
summary.round(3)

In [ ]:
# Lỗi sai dữ kiện tách theo biểu đồ có / không ghi số trên dữ liệu
charts = df[df.source == "synthetic_chart"]
if len(charts):
    display(charts.groupby(["run", "values_labelled"])[["facts_stated", "fact_errors"]].sum())
    display(charts[charts.fact_errors > 0][["run", "id", "caption", "numbers_wrong"]])

## 6. Tải kết quả về máy

In [ ]:
from google.colab import files
files.download(f"{PRED_DIR}/scores_summary.csv")
files.download(f"{PRED_DIR}/scores_per_item.csv")